# Transcribe and evaluate Dutch audio with Qwen3-Omni on Spark

Select **DIY-AI workshop (Python 3.12)** and run all cells in order. Project paths resolve whether the kernel starts in the repository root or in `notebooks/`. Connect to the presenter-provided VPN first. Requires `requests`, `ffmpeg` on PATH, and the Qwen Omni server at `http://spark-0240:8000`.

The UMC Utrecht MP3 is decoded locally to mono 16 kHz WAV and split into chunks of at most 28 seconds, preferring pauses. Each chunk is uploaded to Spark's chat completions endpoint with audio input and text output. Chunking and the transcription prompt match the Gemma notebook; chunks do not overlap and words may be split when no suitable pause is found.

The full transcription is displayed before evaluation. The final cell prints **word error rate (WER)** against both curated references using the same normalization and edit-distance calculation as the Whisper and Gemma notebooks. Reference text is used only for scoring and is never sent to the model.

API reference: [Qwen3-Omni serving](https://docs.vllm.ai/projects/vllm-omni/en/stable/user_guide/examples/online_serving/qwen3_omni/).

Set `QWEN_BASE_URL`, `QWEN_MODEL`, and optionally `QWEN_API_KEY` before starting Jupyter, or edit the setup cell. This is a client notebook for the workshop server, not a local Qwen installation.


**Saved outputs are historical workshop runs, retained as evidence.** They are not a test of your current machine or the selected kernel. Restart the kernel and run all cells to collect your own results. The presentation keeps its dated measurements.


In [1]:
import base64
import io
import os
import re
import shutil
import subprocess
import sys
import time
import wave
from pathlib import Path

import requests

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_URL = (
    os.environ.get("QWEN_BASE_URL", "http://spark-0240:8000")
    .rstrip("/")
    .removesuffix("/v1")
)
API_KEY = os.environ.get("QWEN_API_KEY", "")
headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
MODEL_NAME = os.environ.get("QWEN_MODEL", "Qwen/Qwen3-Omni-30B-A3B-Instruct")
DATA_DIR = PROJECT_ROOT / "data/UMC Utrecht - Jeroen Hendrikse - radiologieverslag"
AUDIO_PATH = DATA_DIR / f"{DATA_DIR.name}.mp3"
CURATED_TXT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.txt"
CURATED_SRT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.srt"
CHUNK_SECONDS = 28
MAX_TOKENS = 1024  # Output budget per chunk; truncated responses raise an error.
TRANSCRIPTION_PROMPT = (
    "Transcribe the following Dutch speech segment verbatim into Dutch text. "
    "Return only the transcription, without explanations, headings, or timestamps. "
    "Preserve repetitions, hesitations, and filler words. Do not summarize or translate."
)

for path in (AUDIO_PATH, CURATED_TXT_PATH, CURATED_SRT_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}. "
            "Download the workshop audio and curated TXT/SRT files and place them "
            "as shown in README.md, section 6 (Audio files)."
        )
if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "Install ffmpeg and make it available on PATH before running this notebook."
    )
if not 0 < CHUNK_SECONDS <= 28:
    raise ValueError("CHUNK_SECONDS must be between 0 and 28 seconds.")

In [2]:
response = requests.get(f"{BASE_URL}/v1/models", headers=headers, timeout=10)
response.raise_for_status()
models = response.json()["data"]
if MODEL_NAME not in [model["id"] for model in models]:
    raise RuntimeError(
        f"{MODEL_NAME} is not served at {BASE_URL}. Available models: {models}"
    )
print(f"Server: {BASE_URL}")
print(f"Model: {MODEL_NAME}")

Server: http://spark-0240:8000
Model: Qwen/Qwen3-Omni-30B-A3B-Instruct


In [3]:
# Decode and detect pauses locally; inference runs on spark-0240.
decoded = subprocess.run(
    [
        "ffmpeg",
        "-hide_banner",
        "-nostdin",
        "-i",
        str(AUDIO_PATH),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-af",
        "silencedetect=noise=-35dB:d=0.2",
        "-f",
        "wav",
        "-acodec",
        "pcm_s16le",
        "pipe:1",
    ],
    check=False,
    capture_output=True,
)
if decoded.returncode:
    raise RuntimeError(decoded.stderr.decode(errors="replace"))

# ffmpeg logs each silence's end and duration; split at its midpoint.
pauses = [
    float(end) - float(duration) / 2
    for end, duration in re.findall(
        r"silence_end: ([0-9.]+) \| silence_duration: ([0-9.]+)",
        decoded.stderr.decode(errors="replace"),
    )
]
with wave.open(io.BytesIO(decoded.stdout), "rb") as audio:
    sample_rate = audio.getframerate()
    sample_width = audio.getsampwidth()
    pcm = audio.readframes(audio.getnframes())
frame_count = len(pcm) // sample_width
max_frames = int(CHUNK_SECONDS * sample_rate)
chunks = []
start = 0
while start < frame_count:
    end = min(start + max_frames, frame_count)
    if end < frame_count:
        candidates = [
            int(p * sample_rate)
            for p in pauses
            if start + max_frames * 0.7 < p * sample_rate <= end
        ]
        if candidates:
            end = max(candidates)
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as chunk:
        chunk.setnchannels(1)
        chunk.setsampwidth(sample_width)
        chunk.setframerate(sample_rate)
        chunk.writeframes(pcm[start * sample_width : end * sample_width])
    chunks.append(
        {
            "start": start / sample_rate,
            "end": end / sample_rate,
            "audio": buffer.getvalue(),
        }
    )
    start = end
print(f"Audio: {frame_count / sample_rate:.1f}s; {len(chunks)} chunks", flush=True)

chunk_results = []
transcription_started = time.perf_counter()
for index, chunk in enumerate(chunks, start=1):
    encoded_audio = base64.b64encode(chunk["audio"]).decode("ascii")
    started = time.perf_counter()
    response = requests.post(
        f"{BASE_URL}/v1/chat/completions",
        headers=headers,
        json={
            "model": MODEL_NAME,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": TRANSCRIPTION_PROMPT},
                        {
                            "type": "audio_url",
                            "audio_url": {
                                "url": f"data:audio/wav;base64,{encoded_audio}"
                            },
                        },
                    ],
                }
            ],
            "modalities": ["text"],
            "temperature": 0,
            "max_tokens": MAX_TOKENS,
        },
        timeout=(10, 600),
    )
    if not response.ok:
        raise RuntimeError(
            f"Chunk {index}: HTTP {response.status_code}: {response.text[:2000]}"
        )
    completion = response.json()
    choice = completion["choices"][0]
    if choice.get("finish_reason") != "stop":
        raise RuntimeError(
            f"Chunk {index} incomplete: {choice.get('finish_reason')}; check MAX_TOKENS."
        )
    text = (choice["message"].get("content") or "").strip()
    if not text:
        raise RuntimeError(f"Chunk {index} returned no transcription.")
    elapsed = time.perf_counter() - started
    chunk_results.append(
        {
            "start": chunk["start"],
            "end": chunk["end"],
            "text": text,
            "elapsed_seconds": elapsed,
            "usage": completion.get("usage", {}),
        }
    )
    print(
        f"Chunk {index}/{len(chunks)} ({chunk['start']:.1f}–{chunk['end']:.1f}s): {elapsed:.1f}s",
        flush=True,
    )

transcription = " ".join(chunk["text"] for chunk in chunk_results)
print(f"Transcription completed in {time.perf_counter() - transcription_started:.1f}s")

Audio: 473.6s; 18 chunks
Chunk 1/18 (0.0–26.3s): 27.3s
Chunk 2/18 (26.3–48.4s): 3.8s
Chunk 3/18 (48.4–75.4s): 3.6s
Chunk 4/18 (75.4–102.1s): 4.4s
Chunk 5/18 (102.1–129.2s): 4.3s
Chunk 6/18 (129.2–155.9s): 3.5s
Chunk 7/18 (155.9–183.3s): 4.0s
Chunk 8/18 (183.3–209.2s): 4.0s
Chunk 9/18 (209.2–235.3s): 4.0s
Chunk 10/18 (235.3–257.7s): 3.6s
Chunk 11/18 (257.7–285.1s): 5.1s
Chunk 12/18 (285.1–312.6s): 4.7s
Chunk 13/18 (312.6–338.9s): 4.5s
Chunk 14/18 (338.9–365.3s): 4.3s
Chunk 15/18 (365.3–392.6s): 5.0s
Chunk 16/18 (392.6–419.4s): 4.1s
Chunk 17/18 (419.4–446.4s): 4.5s
Chunk 18/18 (446.4–473.6s): 3.4s
Transcription completed in 98.1s


In [4]:
print(transcription)

Ja, dat zou moeten werken. Ja. Oké. Dus wat er eigenlijk, ik kijk in de verslaglijst en ik zie een MRI hersenen staan van een bepaalde aanvrager. Ik weet dat het een hersentumor follow-up waarschijnlijk zal zijn. Dus ik klik die patiënt aan en zie dan inderdaad een follow-up staan van een patiënt met drie maanden geleden ook een scan, dus ik heb even heel snel zie ik het scanlijstje met de verschillende data van van eerdere onderzoeken. Daarin zie je bijvoorbeeld dat er een jaar lang al follow-up wordt gemaakt, dus een jaar geleden was blijkbaar de ontdekking van de tumor met een aantal onderzoeken extern. Ik kijk daar later naar, ik heb het in mijn geheugen al opgeslagen van oké, als ik de uitgangssituatie wil weten, moet ik de scan van een jaar terug. daarnaast hangen nu is eigenlijk alleen relevant follow-up oké verandert er iets ten opzichte van drie maanden terug dat is eigenlijk de uitgangsdatum gewoon de grove data van moment dat ik kan ga kijken dan hang ik ze voor ga ik in pri

## Compare with curated transcriptions

WER = (substitutions + deletions + insertions) / reference word count.
We use minimum word-level edit distance after Unicode normalization, case folding, and removal of punctuation; hyphens split words.
Accents, numbers, repeated words, and fillers are retained. SRT cue numbers and timestamps are excluded.
This measures transcription differences, including any editorial changes in the curated references.


In [5]:
import re
import unicodedata


def srt_text(contents):
    """Extract subtitle text, excluding each cue's index and timing line."""
    cues = re.split(r"\n\s*\n", contents.strip())
    text = []
    for cue in cues:
        lines = cue.splitlines()
        timing = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if timing is None:
            raise ValueError(f"Subtitle cue has no timestamp: {cue!r}")
        text.extend(lines[timing + 1 :])
    return " ".join(text)


def normalize_words(text):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFKC", text).casefold())


def word_errors(reference, hypothesis):
    """Levenshtein alignment; ties prefer fewer substitutions, then deletions."""
    # Each tuple stores (total errors, substitutions, deletions, insertions).
    previous = [(i, 0, 0, i) for i in range(len(hypothesis) + 1)]
    for i, reference_word in enumerate(reference, start=1):
        current = [(i, 0, i, 0)]
        for j, hypothesis_word in enumerate(hypothesis, start=1):
            if reference_word == hypothesis_word:
                current.append(previous[j - 1])
                continue
            cost, substitutions, deletions, insertions = previous[j - 1]
            substitute = (cost + 1, substitutions + 1, deletions, insertions)
            cost, substitutions, deletions, insertions = previous[j]
            delete = (cost + 1, substitutions, deletions + 1, insertions)
            cost, substitutions, deletions, insertions = current[j - 1]
            insert = (cost + 1, substitutions, deletions, insertions + 1)
            current.append(min(substitute, delete, insert))
        previous = current
    errors, substitutions, deletions, insertions = previous[-1]
    if not reference:
        raise ValueError("The reference contains no words; WER is undefined.")
    return {
        "reference_words": len(reference),
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "errors": errors,
        "wer": errors / len(reference),
    }


references = {
    "Curated TXT (primary)": CURATED_TXT_PATH.read_text(encoding="utf-8-sig"),
    "Curated SRT": srt_text(CURATED_SRT_PATH.read_text(encoding="utf-8-sig")),
}

In [6]:
hypothesis_words = normalize_words(transcription)
reference_words = {name: normalize_words(text) for name, text in references.items()}
comparison_results = {
    name: word_errors(words, hypothesis_words)
    for name, words in reference_words.items()
}

print(f"Qwen3-Omni transcription: {len(hypothesis_words)} normalized words")
print(
    f"Curated TXT and SRT match after normalization: {len({tuple(words) for words in reference_words.values()}) == 1}"
)
for name, result in comparison_results.items():
    print(f"\n{name} — Word error rate (WER): {result['wer']:.2%}")
    print(
        f"  ({result['substitutions']} substitutions + "
        f"{result['deletions']} deletions + {result['insertions']} insertions)"
        f" / {result['reference_words']} reference words"
        f" = {result['errors']} / {result['reference_words']}"
    )

Qwen3-Omni transcription: 1361 normalized words
Curated TXT and SRT match after normalization: True

Curated TXT (primary) — Word error rate (WER): 8.44%
  (73 substitutions + 15 deletions + 26 insertions) / 1350 reference words = 114 / 1350

Curated SRT — Word error rate (WER): 8.44%
  (73 substitutions + 15 deletions + 26 insertions) / 1350 reference words = 114 / 1350
